In [1]:
pinyins = open('data/names1_pinyin_components.txt','r').read().splitlines()
zhs = open('data/names1.txt','r').read().splitlines()
data = zip(pinyins[:10],zhs[:10])
for pinyin, zh in data:
    print(f'{zh:5}\t{pinyin}')

倪|子宁 	n i | z i n ing
宋|云深 	s ong | y un sh en
尤|听澜 	y ou | t ing l an
唐|承安 	t ang | ch eng _ an
龙|长宁 	l ong | zh ang n ing
田|清玄 	t ian | q ing x uan
柯|川  	k e | ch uan
郑|灵微 	zh eng | l ing w ei
任|瑶音 	r en | y ao y in
汪|音  	w ang | y in


In [2]:
def tokenize(name):
    return name.split()
tokenized_pinyins = [tokenize(p) for p in pinyins]
tokenized_pinyins[:5]

[['n', 'i', '|', 'z', 'i', 'n', 'ing'],
 ['s', 'ong', '|', 'y', 'un', 'sh', 'en'],
 ['y', 'ou', '|', 't', 'ing', 'l', 'an'],
 ['t', 'ang', '|', 'ch', 'eng', '_', 'an'],
 ['l', 'ong', '|', 'zh', 'ang', 'n', 'ing']]

In [3]:
tokens = sorted(set(t for ts in tokenized_pinyins for t in ts))

stoi = {s: i + 1 for i, s in enumerate(tokens)}
stoi['.'] = 0

itos = {i: s for s, i in stoi.items()}

len(stoi), tokens


(58,
 ['_',
  'a',
  'ai',
  'an',
  'ang',
  'ao',
  'b',
  'c',
  'ch',
  'd',
  'e',
  'ei',
  'en',
  'eng',
  'f',
  'g',
  'h',
  'i',
  'ia',
  'ian',
  'iang',
  'iao',
  'ie',
  'in',
  'ing',
  'iong',
  'iu',
  'j',
  'k',
  'l',
  'm',
  'n',
  'o',
  'ong',
  'ou',
  'p',
  'q',
  'r',
  's',
  'sh',
  't',
  'u',
  'ua',
  'uai',
  'uan',
  'uang',
  'ue',
  'ui',
  'un',
  'uo',
  'v',
  'w',
  'x',
  'y',
  'z',
  'zh',
  '|'])

In [4]:
import torch

N = torch.zeros((len(stoi), len(stoi)), dtype=torch.int32)

for ts in tokenized_pinyins:
    seq = ['.'] + ts + ['.']
    for t1, t2 in zip(seq, seq[1:]):
        ix1 = stoi[t1]
        ix2 = stoi[t2]
        N[ix1, ix2] += 1

N.shape


torch.Size([58, 58])

In [5]:
P = (N + 1).float()
P /= P.sum(1, keepdim=True)


In [6]:
g = torch.Generator().manual_seed(2147483647)

def sample_tokens():
    out = []
    ix = 0
    while True:
        ix = torch.multinomial(P[ix], num_samples=1, replacement=True, generator=g).item()
        tok = itos[ix]
        if tok == '.':
            break
        out.append(tok)
    return out

for _ in range(20):
    print(sample_tokens())


['n', 'ing']
['l', 'iang', 'j', 'iu', '|', 'q', 'ing', 'y', 'un', '|', 'y', 'uan']
['zh', 'ao', '|', 'm', 'a', 'v', '|', 'w', 'an', 'q', 'ing', '|', 'l', 'ong', '|', 'y', 'ao', 'y', 'un', 'ch', 'eng', '|', 'l', 'd', 'ong', '|', 'w', 'ei', 'n', 'ian', '|', 'q', 'uang']
['t', 'ang', 'y', 'in', '|', 'y', 'ao', 'q', 'ing', 'zh', 'i', '|', 'j', 'iao', '|', 'y', 'i', 'y', 'ue']
['zh', 'ao', 'w', 'ang', '|', 'zh', 'uang']
['g', 'uan', 'q', 'ing']
['l', 'in']
['ch', 'e', '|', 'h', 'e']
['zh', 'i', 'b', 'ai', '|', 't', 'ian', 'uai', 'y', 'u']
['h', 'e']
['m', 'ing']
['h', 'ong', '|', 'p', 'an']
['_', 'n', 'ing']
['h', 'an', 'q', 'ing']
['g', 'ui', 'y', 'uan', '|', 'j', 'ing', 'zh', 'ao', 'w', 'u', '|', 'l', 'in', '|', 'zh', 'i']
['k', 'ong', '|', 'j', 'iang', '|', 'r', 'uo', '|', 'y', 'in']
['m', 'o', 'h', 'eng', 'y', 'un']
['t', 'ing']
['s', 'v', '|', 'r', 'en', '|', 'm', 'ing', '|', 'sh', 'en', '|', 'zh', 'ao']
['y', 'an', 'w', 'ei']


In [7]:
def detokenize(tokens):
    return ' '.join(tokens)

for _ in range(20):
    print(detokenize(sample_tokens()))


c i | zh ao e
q ing n ie | y ang | t h ao y ao
zh ang un | w an p ei
n ing
l ong | r uo sh en
zh uang _ an
m a f ei | y ue b l ing y u
l an t ao | y ao
m o | ch en | r y ue
sh ao g uang | y u x ue n an g ui iao | z i m o w ao y ao s ong | n ing
w u
w ei
y un
y ao | y un
sh an
y un | zh ang _ an
l iu w ang | t ing y u | t an w an
t ang
m ing m ing y u
t u | zh ang | j iao


In [8]:
INITIALS = {
    '_', 'b','p','m','f','d','t','n','l','g','k','h',
    'j','q','x','r','z','c','s','y','w','zh','ch','sh'
}

FINALS = {
    'a','o','e','i','u','v',
    'ai','ei','ui','ao','ou','iu','ie','ue','er',
    'an','en','in','un','vn',
    'ang','eng','ing','ong',
    'ia','iao','ian','iang','iong',
    'ua','uo','uai','uan','uang'
}

def is_valid(tokens):
    if tokens.count('|') != 1:
        return False

    sep = tokens.index('|')
    if sep == 0 or sep == len(tokens) - 1:
        return False

    parts = tokens[:sep] + tokens[sep + 1:]
    if len(parts) % 2 != 0:
        return False

    for i in range(0, len(parts), 2):
        if parts[i] not in INITIALS:
            return False
        if parts[i + 1] not in FINALS:
            return False

    return True


In [12]:
samples = []

while len(samples) < 20:
    ts = sample_tokens()
    if is_valid(ts):
        samples.append(ts)

for ts in samples:
    print(detokenize(ts))


p uang | n ing
h an n ing zh ang | zh i
g uo x ie | y a
y u j ia | l iao
zh en | x ue
zh ang | y un
l i m ing f ang | g uan
y i | x uan y uan y uan
d ao g ui | y e
y u | l uan
x ue n ing h an | q iu w ei
c ai | sh en
y ang | y ue q ing
h ong | w ei
l in | r uo x uan m ing
zh ang | m ing h ua j ing
y ue h eng | q ing x uan
l in | x i
h eng | y ao
w ei | t ang


In [19]:
import random
def tokens_to_syllable_keys(tokens):
    if tokens.count('|') != 1:
        return None

    sep = tokens.index('|')
    parts = tokens[:sep] + tokens[sep + 1:]

    if len(parts) % 2 != 0:
        return None

    keys = []
    for i in range(0, len(parts), 2):
        keys.append(parts[i] + ' ' + parts[i + 1])

    return keys

def sample_hanzi_name(tokens, mapping, rng=random):
    keys = tokens_to_syllable_keys(tokens)
    if keys is None:
        return None

    chars = []
    for k in keys:
        choices = mapping.get(k)
        if not choices:
            return None
        chars.append(rng.choice(choices))

    return ''.join(chars)

for ts in samples:
    print(detokenize(ts), '=>', sample_hanzi_name(ts, mapping))

print('\n')    
for ts in samples:
    keys = tokens_to_syllable_keys(ts)
    if keys is None:
        print(detokenize(ts), '=>', '<INVALID>')
        continue

    candidates = [mapping.get(k, ['<UNK>']) for k in keys]
    print(detokenize(ts), '=>', candidates)



p uang | n ing => None
h an n ing zh ang | zh i => 寒宁张执
g uo x ie | y a => 郭谢涯
y u j ia | l iao => 瑜贾廖
zh en | x ue => 真薛
zh ang | y un => 章云
l i m ing f ang | g uan => 李冥方官
y i | x uan y uan y uan => 易轩袁袁
d ao g ui | y e => 道桂叶
y u | l uan => 瑜鸾
x ue n ing h an | q iu w ei => 雪宁含邱魏
c ai | sh en => 蔡申
y ang | y ue q ing => 杨月晴
h ong | w ei => 洪韦
l in | r uo x uan m ing => 林若玄明
zh ang | m ing h ua j ing => 张冥华璟
y ue h eng | q ing x uan => 月衡青玄
l in | x i => 林溪
h eng | y ao => 衡遥
w ei | t ang => 韦唐


p uang | n ing => [['<UNK>'], ['宁']]
h an n ing zh ang | zh i => [['含', '韩', '涵', '寒'], ['宁'], ['长', '章', '张'], ['知', '执']]
g uo x ie | y a => [['郭'], ['谢'], ['涯']]
y u j ia | l iao => [['玉', '宇', '余', '雨', '于', '瑜', '羽', '俞', '尉'], ['贾'], ['廖']]
zh en | x ue => [['甄', '真'], ['雪', '薛']]
zh ang | y un => [['长', '章', '张'], ['云', '韵']]
l i m ing f ang | g uan => [['离', '李', '黎', '里', '厉', '立', '璃'], ['明', '冥'], ['方'], ['观', '管', '关', '官']]
y i | x uan y uan y uan => [['易', '衣'], ['玄', '轩'], ['远

In [20]:
def sample_hanzi_name_with_sep(tokens, mapping, rng=random):
    if tokens.count('|') != 1:
        return None

    sep = tokens.index('|')
    left = tokens[:sep]
    right = tokens[sep + 1:]

    def convert_side(parts):
        if len(parts) % 2 != 0:
            return None
        chars = []
        for i in range(0, len(parts), 2):
            k = parts[i] + ' ' + parts[i + 1]
            choices = mapping.get(k)
            if not choices:
                return None
            chars.append(rng.choice(choices))
        return ''.join(chars)

    surname = convert_side(left)
    given = convert_side(right)

    if surname is None or given is None:
        return None

    return surname + '|' + given
for ts in samples:
    print(detokenize(ts), '=>', sample_hanzi_name_with_sep(ts, mapping))


p uang | n ing => None
h an n ing zh ang | zh i => 含宁章|执
g uo x ie | y a => 郭谢|涯
y u j ia | l iao => 余贾|廖
zh en | x ue => 真|雪
zh ang | y un => 张|韵
l i m ing f ang | g uan => 离明方|观
y i | x uan y uan y uan => 衣|轩远元
d ao g ui | y e => 道归|夜
y u | l uan => 俞|鸾
x ue n ing h an | q iu w ei => 薛宁寒|秋魏
c ai | sh en => 蔡|深
y ang | y ue q ing => 扬|月晴
h ong | w ei => 洪|魏
l in | r uo x uan m ing => 临|若轩冥
zh ang | m ing h ua j ing => 章|冥华璟
y ue h eng | q ing x uan => 岳衡|青轩
l in | x i => 临|西
h eng | y ao => 衡|遥
w ei | t ang => 卫|汤
